In [1]:
# Imports
from pathlib import Path
from experiment.utils import TrainedModel, TrainedModelID

import pandas as pd
import torch
from neuralhydrology.nh_run import start_run, eval_run, finetune
from experiment.eval import evaluate_models

In [2]:
model = TrainedModel(TrainedModelID.SOTA_20)

df = pd.read_csv(model.metrics_file, dtype={'basin':str})

basin_data = df.loc[df['NSE'] < df['NSE'].median()].sample(n=1)

basin = basin_data.basin.iloc[0]

basin = '01350080'

In [3]:
# Add the path to the pre-trained model to the finetune config
with open("finetune.yml", "a") as fp:
    fp.write(f"\nbase_run_dir: {model.run_dir.absolute()}")
    
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [4]:
# finetune
finetune(Path("finetune.yml"))

# Epoch 8: 100%|██████████| 13/13 [00:00<00:00, 26.55it/s, Loss: 0.0042]
2024-09-18 17:27:20,605: Epoch 8 average loss: avg_loss: 0.00630, avg_total_loss: 0.00630
# Epoch 9: 100%|██████████| 13/13 [00:00<00:00, 27.36it/s, Loss: 0.0038]
2024-09-18 17:27:21,086: Epoch 9 average loss: avg_loss: 0.00569, avg_total_loss: 0.00569
# Validation: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:339: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:382: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:339: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:382: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)



2024-09-18 17:27:22,186: Epoch 9 average validation loss: 0.01600 -- Median validation metrics: avg_loss: 0.01600, NSE: 0.75795, KGE: 0.66265, Alpha-NSE: 0.71098, Beta-NSE: -0.06740, MSE: 1.60361, RMSE: 1.26634, Pearson-r: 0.89172, Beta-KGE: 0.86381, FHV: -22.68426, FMS: -24.63644, FLV: 10.60977, Peak-Timing: 0.14286, Missed-Peaks: 0.23529, Peak-MAPE: 35.41148
# Epoch 10: 100%|██████████| 13/13 [00:00<00:00, 26.26it/s, Loss: 0.0039]
2024-09-18 17:27:22,684: Epoch 10 average loss: avg_loss: 0.00609, avg_total_loss: 0.00609


In [ ]:
config_file_path = Path(__file__).parent / 'runs' / 'cudalstm_531_basins_finetuned' / 'config.yml'
finetuned_model = TrainedModel(config_file_path_or_experiment_name=config_file_path)
evaluate_models([model, finetuned_model], basins=[basin])

In [ ]:
# load test results of the base run
df_pretrained = pd.read_csv(run_dir / "test/model_epoch003/test_metrics.csv", dtype={'basin': str})
df_pretrained = df_pretrained.set_index("basin")
    
# load test results of the finetuned model
df_finetuned = pd.read_csv(finetune_dir / "test/model_epoch010/test_metrics.csv", dtype={'basin': str})
df_finetuned = df_finetuned.set_index("basin")
    
# extract basin performance
base_model_nse = df_pretrained.loc[df_pretrained.index == basin, "NSE"].values[0]
finetune_nse = df_finetuned.loc[df_finetuned.index == basin, "NSE"].values[0]
print(f"Basin {basin} base model performance: {base_model_nse:.3f}")
print(f"Performance after finetuning: {finetune_nse:.3f}")

So we see roughly the same performance increase in the test period (slightly higher), which is great. However, note that a) our base model was not optimally trained (we stopped quite early) but also b) the finetuning settings were chosen rather randomly. From our experience so far, you can almost always get performance increases for individual basins with finetuning, but it is difficult to find settings that are universally applicable. However, this tutorial was just a showcase of how easy it actually is to finetune models with the NeuralHydrology library. Now it is up to you to experiment with it.